In [0]:
%run ./_local_config

Connect to storage

In [0]:
from azure.storage.blob import BlobServiceClient
import json
import pandas as pd
import io
import datetime

conn_str = f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};AccountKey={storage_account_key};EndpointSuffix=core.windows.net"
blob_service = BlobServiceClient.from_connection_string(conn_str)

Read the full combined history

In [0]:
blob_client = blob_service.get_blob_client(container="bronze", blob="live/battery_full_history.json")
stream = blob_client.download_blob().readall()
data = json.loads(stream)

bronze = pd.DataFrame(data.get("value", data)) if isinstance(data, dict) else pd.DataFrame(data)

print(bronze.shape)
print(f"Date range: {pd.to_datetime(bronze['postingDate']).min()} to {pd.to_datetime(bronze['postingDate']).max()}")

Trim incomplete "today" from raw bronze

In [0]:
bronze["postingDate"] = pd.to_datetime(bronze["postingDate"])
today = pd.Timestamp(datetime.date.today())

bronze_trimmed = bronze[bronze["postingDate"] < today].copy()

print(f"Before trim: {len(bronze)} rows, max date: {bronze['postingDate'].max()}")
print(f"After trim: {len(bronze_trimmed)} rows, max date: {bronze_trimmed['postingDate'].max()}")

Raw data exploration

In [0]:
print(bronze_trimmed["entryType"].value_counts())
print(bronze_trimmed["documentType"].value_counts())

In [0]:
print(bronze_trimmed["itemCategoryCode"].value_counts(dropna=False))

In [0]:
battery_only = bronze_trimmed[bronze_trimmed["itemCategoryCode"] == "BATTERY"]
print(battery_only["brandCode"].value_counts())

In [0]:
print(bronze_trimmed["countryRegionCode"].value_counts(dropna=False))

In [0]:
print(battery_only["itemCategory2"].value_counts(dropna=False))

In [0]:
print(bronze_trimmed.isnull().sum().sort_values(ascending=False))

In [0]:
print(bronze_trimmed.groupby("documentType")["quantity"].describe())

In [0]:
print(bronze_trimmed.groupby(["locationCode", "locationDescription"]).size().sort_values(ascending=False))

Clean to silver

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.transform.clean_silver import clean_to_silver
from src.transform.build_gold_features import build_gold_overall_weekly, build_gold_overall_monthly

silver = clean_to_silver(bronze_trimmed)
print(silver.shape)
print(f"Date range: {silver['posting_date'].min()} to {silver['posting_date'].max()}")

Year distribution analysis

In [0]:
silver["year"] = silver["posting_date"].dt.year

yearly_stats = silver.groupby("year").agg(
    total_net_units=("net_units", "sum"),
    row_count=("net_units", "count"),
    date_min=("posting_date", "min"),
    date_max=("posting_date", "max")
).reset_index()

print(yearly_stats)

In [0]:
import matplotlib.pyplot as plt

weekly = build_gold_overall_weekly(silver)
weekly["year"] = weekly["week_start"].dt.year

plt.figure(figsize=(16, 6))
for yr in sorted(weekly["year"].unique()):
    subset = weekly[weekly["year"] == yr]
    plt.plot(subset["week_start"], subset["total_units_sold"], label=str(yr))

plt.title("Weekly Net Units Sold, by Year")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
yearly_avg_weekly = weekly.groupby("year")["total_units_sold"].agg(["mean", "std", "count"]).reset_index()
print(yearly_avg_weekly)

In [0]:
weekly["week_of_year"] = weekly["week_start"].dt.isocalendar().week

plt.figure(figsize=(16, 6))
for yr in sorted(weekly["year"].unique()):
    subset = weekly[weekly["year"] == yr].sort_values("week_of_year")
    plt.plot(subset["week_of_year"], subset["total_units_sold"], label=str(yr), marker="o", markersize=3)

plt.title("Weekly Sales by Week-of-Year, Overlaid Across Years")
plt.xlabel("Week of Year")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
gold_weekly = build_gold_overall_weekly(silver)
gold_monthly = build_gold_overall_monthly(silver)

print(gold_weekly.shape)
print(gold_weekly.tail(10)[["week_start", "total_units_sold"]])

print(gold_monthly.shape)
print(gold_monthly.tail(5)[["month_start", "total_units_sold"]])

In [0]:
#%pip install lightgbm prophet

In [0]:
#dbutils.library.restartPython()

In [0]:
feature_cols_weekly = ["week_of_year", "month", "contains_month_end", "lag_4w", "rolling_avg_4w"]
target_col = "total_units_sold"

model_data_weekly = gold_weekly.dropna(subset=feature_cols_weekly + [target_col]).copy()
model_data_weekly = model_data_weekly.sort_values("week_start")

split_idx = int(len(model_data_weekly) * 0.8)
train_weekly = model_data_weekly.iloc[:split_idx]
test_weekly = model_data_weekly.iloc[split_idx:]

print(f"Train: {len(train_weekly)} weeks, Test: {len(test_weekly)} weeks")

X_train_w, y_train_w = train_weekly[feature_cols_weekly], train_weekly[target_col]
X_test_w, y_test_w = test_weekly[feature_cols_weekly], test_weekly[target_col]